# GenieX NPU Inference Examples

This notebook demonstrates how to use GenieX for AI inference tasks on NPU devices, including:

- **LLM (Large Language Model)**: Text generation and conversation
- **VLM (Vision Language Model)**: Multimodal understanding and generation

---

## Prerequisites
> ⚠️ Do **not** use Conda or x86 builds — they are incompatible with native ARM64 binaries. If you are in a conda environment, run `conda deactivate` first.

### Step 1: Check your current Python version

Run the cell below to detect whether you already have a compatible Python installed (ARM64, version 3.11–3.13). If you do, you can skip the download step.

In [1]:
import subprocess, sys

RED = "\033[91m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
BOLD = "\033[1m"
RESET = "\033[0m"

# Check the `python` on PATH (what venv creation will use)
result = subprocess.run(
    ["python", "-c",
     "import sys, platform, struct; "
     "print(sys.version_info.major, sys.version_info.minor, sys.version_info.micro, platform.machine(), struct.calcsize('P')*8)"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print(f"{RED}{BOLD}✗ No 'python' found on PATH.{RESET}")
    print(f"{YELLOW}  → Run Step 2 to install Python ARM64.{RESET}")
else:
    parts = result.stdout.strip().split()
    major, minor, micro = int(parts[0]), int(parts[1]), int(parts[2])
    arch = parts[3]
    bits = parts[4]

    print(f"Python on PATH : {major}.{minor}.{micro}")
    print(f"Architecture   : {arch}")
    print(f"Pointer size   : {bits}-bit")
    print()

    ver_ok = (3, 11) <= (major, minor) < (3, 14)
    arch_ok = arch.lower() in ("arm64", "aarch64")

    if ver_ok and arch_ok:
        print(f"{GREEN}{BOLD}✓ Python is compatible (ARM64, 3.11–3.13). You can skip Step 2.{RESET}")
    else:
        reasons = []
        if not ver_ok:
            reasons.append(f"version {major}.{minor} is outside 3.11–3.13")
        if not arch_ok:
            reasons.append(f"architecture is {arch} (need ARM64)")
        print(f"{RED}{BOLD}✗ Python is NOT compatible: {'; '.join(reasons)}.{RESET}")
        print(f"{YELLOW}  → Run Step 2 to download and install the correct Python.{RESET}")

Python on PATH : 3.13.3
Architecture   : ARM64
Pointer size   : 64-bit

✓ Python is compatible (ARM64, 3.11–3.13). You can skip Step 2.


### Step 2: Install ARM64 Python (only if the check above failed)

**Skip this cell if your Python is already compatible.**

The cell below downloads and launches the Python 3.13 ARM64 installer. During installation:

> **IMPORTANT**: Select **"Add python.exe to PATH"** on the first screen of the installer.

> After installation completes, **restart your terminal / IDE** before continuing.

In [ ]:
!curl --ssl-no-revoke -Lo python-3.13.3-arm64.exe https://www.python.org/ftp/python/3.13.3/python-3.13.3-arm64.exe && start python-3.13.3-arm64.exe

### Step 3: Fix PATH (if needed)

If your environment PATH is overridden by a package manager, run the cell below to restore the system PATH so the new Python is found.

**Troubleshooting** (if `python` still shows AMD64 or wrong version):
- Run `conda deactivate` if you have conda active.
- Open **Edit the system environment variables** → **Environment Variables** → edit `Path` → move ARM64 Python to the top.
- If you forgot "Add python.exe to PATH", re-run the installer and select it.

In [3]:
!powershell -NoProfile -Command "$systemPath = [Environment]::GetEnvironmentVariable('Path', 'Machine'); $userPath = [Environment]::GetEnvironmentVariable('Path', 'User'); [Environment]::SetEnvironmentVariable('Path', \"$userPath;$systemPath\", 'Process')"

### Step 4: Create a virtual environment

> **Note:** `activate` does not persist across notebook cells (each `!` runs in its own subprocess), so we call the venv's `python` and `pip` by their full path in subsequent steps.

In [4]:
!python -m venv geniex-env

In [2]:
import subprocess, sys, platform

result = subprocess.run(
    [r"geniex-env\Scripts\python", "-c",
     "import platform, sys; print(f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro} {platform.machine()}')"],
    capture_output=True, text=True
)
if result.returncode == 0:
    info = result.stdout.strip()
    print(f"Venv Python: {info}")
    if "ARM64" not in info.upper():
        print("\033[91m✗ Venv was created with wrong Python! Delete geniex-env and re-run Step 4 after fixing PATH.\033[0m")
    else:
        print("\033[92m✓ Venv is ARM64. Proceed to Step 5.\033[0m")
else:
    print("\033[91m✗ Venv not found. Run the cell above first.\033[0m")

Venv Python: 3.13.3 ARM64
✓ Venv is ARM64. Proceed to Step 5.


### Step 5: Install GenieX

Install the GenieX package into the virtual environment using the venv's pip directly:

In [7]:
!geniex-env\Scripts\python -m pip install -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple geniex

Looking in indexes: https://test.pypi.org/simple/, https://pypi.org/simple
  Using cached geniex-0.1.2rc4-py3-none-any.whl



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: c:\Users\zackli\zackli\dev-code\geniex\examples\python\geniex-env\Scripts\python.exe -m pip install --upgrade pip


### Step 6: Select the venv as your Jupyter kernel

1. In VS Code / Cursor: click the **kernel picker** (top-right of the notebook).
2. Choose **"Python Environments"** → select `geniex-env` (or browse to `geniex-env\Scripts\python.exe`).
3. The kernel will reload automatically.

---

## Kernel Verification

Run the cell below to confirm the active kernel is ARM64 Python 3.11–3.13:

In [3]:
import sys
import platform

RED = "\033[91m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
BOLD = "\033[1m"
RESET = "\033[0m"

min_ver = (3, 11)
max_ver = (3, 14)
current_ver = sys.version_info
arch = platform.machine().lower()
ver_ok = min_ver <= (current_ver.major, current_ver.minor) < max_ver
arch_ok = arch in ("arm64")

if ver_ok and arch_ok:
    print(f"{GREEN}{BOLD}[VERIFICATION PASSED]{RESET} Python {current_ver.major}.{current_ver.minor}.{current_ver.micro} {platform.machine()}")
    print("You may continue to the examples below.")
else:
    print(f"{RED}{BOLD}[VERIFICATION FAILED]{RESET} Python {current_ver.major}.{current_ver.minor}.{current_ver.micro} {platform.machine()}")
    print(f"{YELLOW}Required: Python 3.11–3.13 ARM64.{RESET}")
    print("Please go back and complete the installation steps above.")
    sys.exit(1)

[VERIFICATION PASSED] Python 3.13.3 ARM64
You may continue to the examples below.


---

## Example 1: LLM (Large Language Model) NPU Inference

Load a model and generate text. This example uses `qwen3` — GenieX will automatically download the model weights on first use.

In [12]:
from geniex import AutoModelForCausalLM

# GGUF Model Example
model = AutoModelForCausalLM.from_pretrained(
    "bartowski/Qwen_Qwen3.5-0.8B-GGUF",
     quant="Q4_0",
    device_map="llama_cpp"
)

messages = [{"role": "user", "content": "What is 2+2?"}]
prompt = model.tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
)

# One-shot
output = model.generate(prompt, max_new_tokens=256)
print(output.text)
print(f"[{output.profile.generated_tokens} tok, "
      f"{output.profile.decode_speed:.1f} tok/s, stop={output.profile.stop_reason}]")

# Streaming
streamer = model.generate(prompt, max_new_tokens=256, stream=True)
for chunk in streamer:
    print(chunk, end="", flush=True)

model.close()

**2 + 2 = 4**

In mathematics and arithmetic, the operation of adding two numbers results in their sum. Here, we add four units together to make five, which equals four. Therefore, **4**.
[46 tok, 10.6 tok/s, stop=eos]


---

## Example 2: VLM (Vision Language Model) NPU Inference

Load a vision-language model and describe an image. Make sure `demo.jpeg` exists in the same directory as this notebook (or update the path below).

In [26]:
!curl --ssl-no-revoke -Lo demo.jpeg https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  2 484.7k   2  11737   0      0  10971      0   00:45   00:01   00:44  10989
 92 484.7k  92 447.4k   0      0 206.7k      0   00:02   00:02         206.9k
100 484.7k 100 484.7k   0      0 223.7k      0   00:02   00:02         206.9k
100 484.7k 100 484.7k   0      0 223.7k      0   00:02   00:02         206.9k
100 484.7k 100 484.7k   0      0 223.7k      0   00:02   00:02         206.9k


In [ ]:
import os
from geniex import AutoModelForVision2Seq

image_path = os.path.abspath("demo.jpeg")

# Optionally, replace with a local .gguf / .bin path
model = AutoModelForVision2Seq.from_pretrained("ggml-org/SmolVLM-500M-Instruct-GGUF", quant="Q8_0", device_map="llama_cpp")
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image_path},
        {"type": "text", "text": "Describe the image."},
    ],
}]
prompt = model.tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
)
print(model.generate(prompt, images=[image_path]).text)

Image path: c:\Users\zackli\zackli\dev-code\geniex\examples\python\demo.jpeg
